In [ ]:
from pathlib import Path
import json
import warnings
import sys
import pickle

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation
from visualization.src.utils import BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS, ARCHITECUTURE_FAMILY_COLORS, READOUT_COLORS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:
repo_dir = Path("../../..")
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"

In [ ]:
results_file = artifacts_dir / "mapping_results.csv"
df_all = pd.read_csv(results_file)

In [ ]:
df_all.columns

## Load Experiment Configuration

In [ ]:
analysis_dir = repo_dir / 'analysis'
config_dir = analysis_dir / 'curve_fitting/configs/mapping'
results_dir = analysis_dir / 'curve_fitting/outputs/fitting_results'
# results_dir = analysis_dir / 'curve_fitting/outputs/fitting_results_mapping'

In [ ]:
df_all.probe_type.value_counts()

In [ ]:
datasets = [
    'tvsd',
    'nsd',
    'things_fmri',
    'things_eeg1',
    'things_eeg2',
    'things_meg',
]

mapping_types = [
    'linear',
    'attention_individual',
    'attention_shared',
]

mapping_names = {
    'linear': 'Linear (Single-Subject)',
    'attention_individual': 'Attention (Single-Subject)',
    'attention_shared': 'Attention (Multi-Subject)',
}

selected_mapping_types = ['linear_individual', 'attention_individual', 'attention_shared']

METRIC = 'pearsonr_nc'

In [ ]:
all_configs = {}

for ds in datasets:
    for m in mapping_types:
        yaml_config = config_dir / f'{m}/{ds}.yaml'
        all_configs[f"{ds}-{m}"] = load_yaml(yaml_config)

In [ ]:
L_fit_dict = {key: config['fitting_parameters']['loss_function'] for key, config in all_configs.items()}
L_viz_dict = {key: config['visualization']['loss_function'] for key, config in all_configs.items()}
x_scale_dict = {key: float(config['fitting_parameters']['X_scaler']) for key, config in all_configs.items()}



## Apply Data Filters

In [ ]:
all_df = {
    name: apply_filters(df_all, config.get('data_filters', {}))
    for name, config in all_configs.items()
}

## Load Fitting Results

In [ ]:
optimized_params_dict = {}
opt_params_boot_dict = {}

for c in all_configs.keys():
    with open(results_dir / f'mapping-{c}' / 'results.pkl', 'rb') as f:
        results = pickle.load(f)

    L_fit = L_fit_dict[c]
    L_viz = L_viz_dict[c]
    optimized_params_dict[c] = convert_loss_parameters(results['optimized_parameters'], L_fit, L_viz)

    # Convert bootstrapped parameters
    opt_params_boot = results['optimized_parameters_bootstrapped']
    opt_params_boot_dict[c] = convert_loss_parameters_batch(
        params=opt_params_boot,
        src_loss=L_fit,
        dst_loss=L_viz
    )

In [ ]:
df_all.probe_type.value_counts()

In [ ]:
df_all

# Mapping Scaling: No Fit

In [ ]:
subfigsize = (4, 3)
nrows, ncols = 2, 3

# subfigsize = (8, 3)
# nrows, ncols = 3, 2

zoom = 1.0
figsize = (subfigsize[0] * zoom * ncols, subfigsize[1] * zoom * nrows)

fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=figsize, 
    # sharey=True,
    dpi = 300
    )

benchmark_names = {
    'tvsd': 'TVSD',
    'things_fmri': 'THINGS fMRI',
    'nsd_func1pt8mm_individualROIs': 'NSD fMRI',
    'things_eeg1': 'THINGS EEG1',
    'things_eeg2': 'THINGS EEG2',
    'things_meg': 'THINGS MEG',
}

metric = "pearsonr_nc"
for i, (benchmark_k, benchmark_v) in enumerate(benchmark_names.items()):
    ax = axes[i // ncols, i % ncols]
    data = df_all[df_all['benchmark_name'] == benchmark_k]
    data = data[data['probe_type'].isin(selected_mapping_types)]
    data = data.groupby(
        ['fitting_samples', 'probe_name', 'seed']
    ).agg(
        {metric: 'mean'}
    ).reset_index()
    sns.lineplot(
        data=data,
        x='fitting_samples',
        y=metric,
        hue='probe_name',
        errorbar='sd',
        # errorbar=None,
        marker='o',
        ax=ax,
        palette={v: READOUT_COLORS[k] for k, v in mapping_names.items()},
    )
    
    
    
    ax.set_title(benchmark_v, fontsize=16, fontweight='bold')
    ax.set_xlabel('Fitting samples', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    
    if i == 5:
        ax.legend(title='Probe Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        ax.legend().remove()
    
    
    
plt.tight_layout()

# Mapping Scaling: Linear Fits

In [ ]:
subfigsize = (4, 3)
nrows, ncols = 2, 3

# subfigsize = (8, 3)
# nrows, ncols = 3, 2

zoom = 1.0
figsize = (subfigsize[0] * zoom * ncols, subfigsize[1] * zoom * nrows)

fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=figsize, 
    # sharey=True,
    dpi = 300
    )

benchmark_names = {
    'tvsd': 'TVSD',
    'things_fmri': 'THINGS fMRI',
    'nsd_func1pt8mm_individualROIs': 'NSD fMRI',
    'things_eeg1': 'THINGS EEG1',
    'things_eeg2': 'THINGS EEG2',
    'things_meg': 'THINGS MEG',
}

metric = "pearsonr_nc"
for i, (benchmark_k, benchmark_v) in enumerate(benchmark_names.items()):
    ax = axes[i // ncols, i % ncols]
    data = df_all[df_all['benchmark_name'] == benchmark_k]
    data = data[data['probe_type'].isin(selected_mapping_types)]
    data = data.groupby(
        ['fitting_samples', 'probe_name', 'seed']
    ).agg(
        {metric: 'mean'}
    ).reset_index()
    
    # data = data[data.data_pct > 5]
    # sns.lineplot(
    #     data=data,
    #     x='data_pct',
    #     y=metric,
    #     hue='probe_type',
    #     # errorbar='sd',
    #     errorbar=None,
    #     marker='o',
    #     ax=ax
    # )
    
    # Fit a linear regression line for each probe type
    probe_names = data['probe_name'].unique()
    for probe_name in probe_names:
        subset = data[data['probe_name'] == probe_name]
        # Aggregate by data_pct
        
        agg = subset.groupby(['fitting_samples', 'probe_name', 'seed'])[metric].agg(['mean', 'std', 'count']).reset_index()
        agg['sem'] = agg['std'] / np.sqrt(agg['count'])
        agg['ci95'] = 1.96 * agg['sem']
        
        # ax.errorbar(
        #     agg['fitting_samples'], 
        #     agg['mean'], 
        #     yerr=agg['ci95'], 
        #     label=probe_name,
        #     marker='o',
        #     capsize=5
        # )
        
        color = {v: READOUT_COLORS[k] for k, v in mapping_names.items()}[probe_name]
        
        # Fit linear
        slope, intercept, r_value, p_value, std_err = stats.linregress(np.log(agg['fitting_samples']), agg['mean'])
        x_fit = np.linspace(agg['fitting_samples'].min(), agg['fitting_samples'].max(), 100)
        y_fit = intercept + slope * np.log(x_fit)
        sns.lineplot(x=x_fit, y=y_fit, linestyle='--', ax=ax, label=f"{probe_name}", color=color, linewidth=2.0)
        sns.scatterplot(x=agg['fitting_samples'], y=agg['mean'], s=50, ax=ax, marker='o', alpha=0.3, color=color)
    
    
    ax.set_title(benchmark_v, fontsize=16, fontweight='bold')
    ax.set_xlabel('Fitting samples', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    ax.spines[['top', 'right']].set_visible(False)
    
    if i == 5:
        ax.legend(title='Probe Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        ax.legend().remove()
    
    # ax.legend().remove()
    
    ax.set_ylim(0, 1.0)
    
plt.tight_layout()

# save_figs(fig, save_dir, "fig6_probes_performance_vs_fitting_samples_with_regression_lines", dpi=300, formats=("png", "pdf", "svg"))

## Mapping Scaling: Parametric Fits

In [ ]:
x_extend = 15
X_str = r'$$\tilde{F}$$'
# X_str = r'$$F$$'
linewidth = 3.0
alpha_scatter = 0.2
alpha_ci = 0.2
alpha_fit = 1.0
fig_multiplier = 0.75
# fig_multiplier = 1.0
figsize = (10, 8)
figsize = (10, 7)
figsize = (22, 10)
figsize = (fig_multiplier * figsize[0], fig_multiplier * figsize[1])


percentile_ci = 95


In [ ]:
READOUT_COLORS

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=figsize, dpi=300)
for idx, ds in enumerate(datasets):
    ax = axes.flatten()[idx]

    for j, m in enumerate(mapping_types):
        exp_name = f'{ds}-{m}'
        print(f"Plotting {exp_name}...")
        ### Neuro
        df_region = all_df[exp_name]
        optimized_params_neuro = optimized_params_dict[exp_name]
        opt_params_boot_neuro = opt_params_boot_dict[exp_name]
        L = LOSS_FUNCTIONS[L_viz_dict[exp_name]]
        x_scaler = x_scale_dict[exp_name]
        X = df_region.fitting_samples.values / x_scaler
        
        color = READOUT_COLORS[m]
        # color = color_palette[j]
        # alpha_scatter = 1.0
        sns.scatterplot(data=df_region, x='fitting_samples', y='pearsonr_nc', ax=ax, color=color, alpha=alpha_scatter)
        plot_reg(X, optimized_params_neuro, L, ax, color=color, x_extend=x_extend, linestyle='-', X_str=X_str, x_scaler=x_scaler, show_x_scaler=False, linewidth=linewidth, legend=False, alpha=alpha_fit)
        # plot_reg(X, optimized_params_neuro, L, ax, color=color, x_extend=x_extend, linestyle='-', X_str=X_str, x_scaler=x_scaler, show_x_scaler=False, linewidth=linewidth, alpha=alpha_fit)
        plot_confidence_intervals(X, opt_params_boot_neuro, L, ax, color=color, x_scaler=x_scaler, x_extend=x_extend, alpha=0.1, percentile=percentile_ci, invert_y=True)


    ### Formatting
    ax.set_xscale('log')
    ax.set_xlabel('Neural Samples (F)', fontsize=16, fontweight='bold')
    ax.set_ylabel('Alignment Score (S)', fontsize=16, fontweight='bold')
    ax.set_title(BENCHMARK_NAME_MAPPING[ds], fontsize=20, fontweight='bold')
    # ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
    ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0], minor_grid=False)

    # ### Legend
    handles, labels = ax.get_legend_handles_labels()
    # labels = [
    #     'Linear                      ' + labels[0],
    #     'Attention Individual  '  + labels[1],
    #     'Attention Shared     '  + labels[2]
    # ]
    labels = [
        'Linear               ' + labels[0],
        'Attention           '  + labels[1],
        'Attention-Multi  '  + labels[2]
    ]
    ax.legend(handles, labels, fontsize=12)
    
    # ax.legend().remove()

    ax.spines[['right', 'top']].set_visible(False)
    
plt.tight_layout()


# figures_dir = save_dir
# fig_name = 'mapping_scaling'
# formats = ['pdf', 'png', 'svg']
# save_figs(figures_dir, fig_name, formats=formats)


figures_dir = save_dir
fig_name = 'mapping-parametric_fit-all_readouts'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)


# Readout Scatter Plots: Mapping Parameters vs. Readout Performance

In [ ]:
mapping_params_per_roi_file = artifacts_dir / 'mapping_params_per_roi.csv'
df_mapping_params_per_roi = pd.read_csv(mapping_params_per_roi_file)

df_mapping_params_per_roi.loc[df_mapping_params_per_roi['probe_type'] == 'attn_probe_individual', 'probe_type'] = 'attention_individual'
df_mapping_params_per_roi.loc[df_mapping_params_per_roi['probe_type'] == 'attn_probe_shared', 'probe_type'] = 'attention_shared'

df_mapping_params_per_roi.shape

In [ ]:
df_mapping_params_per_roi

In [ ]:
df = df_all.copy()
df = df[df['data_pct'] == 100]
df = apply_hiearchical_aggregation(
    df=df,
    groupby_cols=['benchmark_name', 'roi', 'probe_type', 'probe_subj_config_name', 'probe_name', 'seed'],
    agg_cols=['pearsonr_nc'],
    agg_func='mean'
)


df_readouts_merged = df.merge(
    df_mapping_params_per_roi,
    on=['benchmark_name', 'roi', 'probe_type'],
    how='left'
)
df_readouts_merged.shape

In [ ]:
df_readouts_merged.probe_type.value_counts()

In [ ]:
# --- your existing code up to the plot ---

zoom = 0.75
figsize = (10, 8)
figsize = (figsize[0] * zoom, figsize[1] * zoom)

fig, axes = plt.subplots(1, 1, figsize=figsize, dpi=300)
ax = axes
df = df_readouts_merged.sort_values(by='probe_name')
df = df[df['probe_type'].isin(selected_mapping_types)]


sns.scatterplot(
    data=df,
    x='param_count',
    y='pearsonr_nc',
    hue='probe_name',
    style='probe_name',
    s=300,
    alpha=0.2,
    palette={v: READOUT_COLORS[k] for k, v in mapping_names.items()},
    ax=ax
)

# --- centroids per probe_type ---
centroids = (
    df.groupby('probe_name', as_index=False)
      .agg(param_count=('param_count', lambda x: np.mean(np.log10(x))),
           pearsonr_nc=('pearsonr_nc', 'mean'))
)
centroids = centroids.sort_values(by='probe_name')
centroids['param_count'] = 10 ** centroids['param_count']

# plot centroid markers (bigger + outlined)
sns.scatterplot(
    data=centroids,
    x='param_count',
    y='pearsonr_nc',
    hue='probe_name',
    style='probe_name',
    s=900,
    alpha=1.0,
    linewidth=2,
    edgecolor='black',
    palette={v: READOUT_COLORS[k] for k, v in mapping_names.items()},
    legend=False,   # don't duplicate legend
    ax=ax
)

# # optional: label centroid points
# for _, r in centroids.iterrows():
#     ax.annotate(
#         str(r['probe_type']),
#         (r['param_count'], r['pearsonr_nc']),
#         textcoords="offset points",
#         xytext=(8, 6),
#         ha='left',
#         fontsize=11,
#         fontweight='bold',
#     )

ax.legend(title='Probe Type',loc='upper right')

leg = ax.get_legend()

handles = leg.legend_handles if hasattr(leg, "legend_handles") else ax.get_legend_handles_labels()[0]

for h in handles:
    if hasattr(h, "set_alpha"):
        h.set_alpha(1.0)


ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

ax.set_xscale('log')
ax.set_xlabel('Number of Mapping Parameters', fontsize=16, fontweight='bold')
ax.set_ylabel('Alignment Score', fontsize=16, fontweight='bold')
# ax.set_title('Mapping Parameters vs Alignment Score', fontsize=20, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()

# figures_dir = save_dir
# fig_name = 'fig7_mapping_parameters_vs_alignment_score'
# formats = ['pdf', 'png', 'svg']
# save_figs(figures_dir, fig_name, formats=formats)


figures_dir = save_dir
fig_name = 'mapping-mapping_parameters_vs_alignment_score'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)

In [ ]:
centroids

In [ ]:
centroids.to_dict()

In [ ]:
probe_relabel = {
    'Shallow_mlp (Single-Subject)': 'Shallow MLP (Single-Subject)',
    'Shallow_mlp (Multi-Subject)': 'Shallow MLP (Multi-Subject)',
    'Low_rank (Single-Subject)': 'Low-rank (Single-Subject)',
    'Low_rank (Multi-Subject)': 'Low-rank (Multi-Subject)',
}


In [ ]:
palette = {
    'Linear (Single-Subject)': READOUT_COLORS['linear'],
    'Attention (Single-Subject)': READOUT_COLORS['attention_individual'],
    'Attention (Multi-Subject)': READOUT_COLORS['attention_shared'],
    'Shallow MLP (Single-Subject)': '#118ab2',
    'Shallow MLP (Multi-Subject)': '#073b4c',
    'Low-rank (Single-Subject)': '#9b5de5',
    'Low-rank (Multi-Subject)': '#5a189a',
}

METRIC = 'pearsonr_nc'

In [ ]:

zoom = 0.75
figsize = (10, 8)
figsize = (figsize[0] * zoom, figsize[1] * zoom)

fig, axes = plt.subplots(1, 1, figsize=figsize, dpi=300)
ax = axes
df = df_readouts_merged.sort_values(by='probe_name')

# relabel probe names for better legend display
df['probe_name'] = df['probe_name'].replace(probe_relabel)

sns.scatterplot(
    data=df,
    x='param_count',
    y='pearsonr_nc',
    hue='probe_name',
    style='probe_name',
    s=300,
    alpha=0.2,
    palette=palette,
    ax=ax
)

# --- centroids per probe_type ---
centroids = (
    df.groupby('probe_name', as_index=False)
      .agg(param_count=('param_count', lambda x: np.mean(np.log10(x))),
           pearsonr_nc=('pearsonr_nc', 'mean'))
)
centroids = centroids.sort_values(by='probe_name')
centroids['param_count'] = 10 ** centroids['param_count']

# plot centroid markers (bigger + outlined)
sns.scatterplot(
    data=centroids,
    x='param_count',
    y='pearsonr_nc',
    hue='probe_name',
    style='probe_name',
    s=500,
    alpha=1.0,
    linewidth=2,
    edgecolor='black',
    palette=palette,
    legend=False,   # don't duplicate legend
    ax=ax
)

# # optional: label centroid points
# for _, r in centroids.iterrows():
#     ax.annotate(
#         str(r['probe_type']),
#         (r['param_count'], r['pearsonr_nc']),
#         textcoords="offset points",
#         xytext=(8, 6),
#         ha='left',
#         fontsize=11,
#         fontweight='bold',
#     )

ax.legend(title='Probe Type',loc='upper right')

leg = ax.get_legend()

handles = leg.legend_handles if hasattr(leg, "legend_handles") else ax.get_legend_handles_labels()[0]

for h in handles:
    if hasattr(h, "set_alpha"):
        h.set_alpha(1.0)


ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

ax.set_xscale('log')
ax.set_xlabel('Number of Mapping Parameters', fontsize=16, fontweight='bold')
ax.set_ylabel('Alignment Score', fontsize=16, fontweight='bold')
# ax.set_title('Mapping Parameters vs Alignment Score', fontsize=20, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()

# figures_dir = save_dir
# fig_name = 'fig7_mapping_parameters_vs_alignment_score'
# formats = ['pdf', 'png', 'svg']
# save_figs(figures_dir, fig_name, formats=formats)


figures_dir = save_dir
fig_name = 'mapping-mapping_parameters_vs_alignment_score'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)

In [ ]:
centroids

In [ ]:

# Build readout comparison table and render as Markdown
from IPython.display import Markdown, display

metric = 'pearsonr_nc'
benchmark_order = ['tvsd', 'nsd_func1pt8mm_individualROIs', 'things_fmri', 'things_eeg1', 'things_eeg2', 'things_meg']
probe_order = [
    'Linear (Single-Subject)',
    'Shallow_mlp (Single-Subject)',
    'Shallow_mlp (Multi-Subject)',
    'Low_rank (Single-Subject)',
    'Attention (Single-Subject)',
    'Attention (Multi-Subject)',
]
probe_label_map = {
    'Linear (Single-Subject)': 'Linear (SS)',
    'Attention (Single-Subject)': 'Attention (SS)',
    'Attention (Multi-Subject)': 'Attention (MS)',
    'Shallow_mlp (Single-Subject)': 'Shallow MLP (SS)',
    'Shallow_mlp (Multi-Subject)': 'Shallow MLP (MS)',
    'Low_rank (Single-Subject)': 'Low-rank (SS)',
}

df_table = df_readouts_merged.loc[df_readouts_merged['probe_name'].isin(probe_order)].copy()

score_summary = (
    df_table
    .groupby(['benchmark_name', 'probe_name'], as_index=False)
    .agg(score=(metric, 'mean'))
)

pivot_scores = score_summary.pivot(index='benchmark_name', columns='probe_name', values='score').reindex(benchmark_order)
pivot_scores.index = pivot_scores.index.map(BENCHMARK_NAME_MAPPING)
pivot_scores = pivot_scores.rename(columns=probe_label_map)

column_order = [probe_label_map[p] for p in probe_order]
pivot_scores = pivot_scores.reindex(columns=column_order)

avg_row = pd.DataFrame([pivot_scores.mean(axis=0)], index=['Average'])
scores_full = pd.concat([pivot_scores, avg_row])

param_avg = (
    df_table
    .groupby('probe_name', as_index=True)['param_count']
    .agg(lambda x: 10 ** np.mean(np.log10(x)))
    .reindex(probe_order)
    .rename(index=probe_label_map)
    .reindex(column_order)
)

param_exp = int(np.floor(np.log10(param_avg.dropna())).min())

def fmt_scaled_md(x, exp):
    if pd.isna(x):
        return 'NA'
    return f'${x / (10 ** exp):.2f}$'

def mark_best(values, mode='max'):
    target = values.max(skipna=True) if mode == 'max' else values.min(skipna=True)
    out = []
    for v in values:
        if pd.isna(v):
            out.append(False)
        else:
            out.append(np.isclose(v, target))
    return out

headers = ['Benchmark'] + list(scores_full.columns)
lines = [
    '| ' + ' | '.join(headers) + ' |',
    '| ' + ' | '.join(['---'] * len(headers)) + ' |',
]

for idx, row in scores_full.iterrows():
    is_best = mark_best(row, mode='max')
    vals = []
    for v, best in zip(row.values, is_best):
        cell = 'NA' if pd.isna(v) else f'{v:.3f}'
        vals.append(f'**{cell}**' if best and cell != 'NA' else cell)
    lines.append('| ' + ' | '.join([idx] + vals) + ' |')

param_best = mark_best(param_avg, mode='min')
param_vals = []
for v, best in zip(param_avg.values, param_best):
    cell = fmt_scaled_md(v, param_exp)
    param_vals.append(f'**{cell}**' if best and cell != 'NA' else cell)
lines.append('| Geom. #Parameters ($\\times 10^{' + str(param_exp) + '}$) | ' + ' | '.join(param_vals) + ' |')

display(Markdown('\n'.join(lines)))


In [ ]:
print("\n".join(lines))

In [ ]:

# Render the same readout comparison table as LaTeX
caption = (
    r'Noise-ceiled Pearson $r$ across benchmarks for different mapping readouts. '
    r'SS = single-subject; MS = multi-subject. Scores are averaged across subjects, ROIs, and three random seeds. '
    r'Parameter counts are summarized with the geometric mean across all instantiations of each probe type. Best per row is shown in \textbf{bold}.'
)
label = 'tab:readout_comparison-extended'

def fmt_scaled_latex(x, exp):
    if pd.isna(x):
        return 'NA'
    return f'{x / (10 ** exp):.2f}'

n_cols = len(scores_full.columns)
col_spec = 'l|' + 'c' * n_cols
header_cells = [rf'\textbf{{{col}}}' for col in scores_full.columns]
row_end = r' \\'

latex_lines = [
    r'\begin{table}[t]',
    r'  \centering',
    r'  \small',
    rf'  \caption{{{caption}}}',
    rf'  \label{{{label}}}',
    r'  \resizebox{\linewidth}{!}{',
    rf'  \begin{{tabular}}{{{col_spec}}}',
    r'    \toprule',
    '    \\textbf{Benchmark} & ' + ' & '.join(header_cells) + row_end,
    r'    \midrule',
]

for idx, row in scores_full.iterrows():
    row_max = row.max(skipna=True)
    rendered = []
    for v in row.values:
        if pd.isna(v):
            rendered.append('NA')
        elif np.isclose(v, row_max):
            rendered.append(rf'\textbf{{{v:.3f}}}')
        else:
            rendered.append(f'{v:.3f}')
    if idx == 'Average':
        latex_lines.append(r'    \midrule')
    latex_lines.append(f"    {idx} & " + ' & '.join(rendered) + row_end)

param_min = param_avg.min(skipna=True)
param_cells = []
for v in param_avg.values:
    if pd.isna(v):
        param_cells.append('NA')
    else:
        c = fmt_scaled_latex(v, param_exp)
        param_cells.append(rf'\textbf{{{c}}}' if np.isclose(v, param_min) else c)

latex_lines.extend([
    r'    \midrule',
    '    Geom. \#Parameters ($\\times 10^{' + str(param_exp) + '}$) & ' + ' & '.join(param_cells) + row_end,
    r'    \bottomrule',
    r'  \end{tabular}',
    r'  }',
    r'\end{table}',
])

print('\n'.join(latex_lines))


## Additional Readout Baselines

To compare attention-based mappings with simpler alternatives, we evaluated a factorized linear (low-rank) readout and a shallow MLP readout. Both operate on the mean-pooled encoded representation: the low-rank readout uses two linear projections, while the shallow MLP adds one nonlinear hidden layer before the subject-specific neural prediction layer.

## Readout Implementation Details

For the low-rank and shallow MLP readouts, given encoded tokens $\{\mathbf{x}_t\}_{t=1}^{T}$ with width $d$, both readouts first compute
$$
\bar{\mathbf{x}} = \frac{1}{T}\sum_{t=1}^{T}\mathbf{x}_t \in \mathbb{R}^{d}.
$$

**Low-rank readout.** For subject $s$ with $V_s$ recorded units, the prediction is produced by a factorized linear map,
$$
\hat{\mathbf{y}}_s = \mathbf{B}_s\mathbf{A}_s\bar{\mathbf{x}} + \mathbf{b}_s, \qquad
\mathbf{A}_s \in \mathbb{R}^{r \times d},\quad \mathbf{B}_s \in \mathbb{R}^{V_s \times r},,
$$
where the first projection has no bias and the second includes the subject-specific bias $\mathbf{b}_s$.

**Shallow MLP readout.** Mean-pooled features are transformed by a one-hidden-layer MLP before a subject-specific linear output head:
$$
\mathbf{z} = \mathbf{W}_2\,\operatorname{Dropout}_{p}\!\left(\operatorname{GELU}\!\left(\mathbf{W}_1\operatorname{LN}(\bar{\mathbf{x}})+\mathbf{b}_1\right)\right)+\mathbf{b}_2,
\qquad
\hat{\mathbf{y}}_s = \mathbf{C}_s\mathbf{z}+\mathbf{c}_s.
$$
The dropout probability $p$ is selected per benchmark configuration.